# GC Example 17: Harris vs Baumgardt Distance Comparison

**EPS Research — Milky Way GC Corpus v1.4.0**

Harris (1996/2010) and Baumgardt (2023) provide independent distance estimates.
Systematic offsets reveal calibration differences.

**Corpus:** Flynn (2026), EPS Research Milky Way Globular Cluster Corpus v1.4.0  
**Sources:** Harris (1996/2010), Vasiliev & Baumgardt (2021), Baumgardt et al. (2023), Schiavon et al. (2024) APOGEE DR17  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corrected GC corpus ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    GC_PATH = 'harris_gc_corpus_v1.4.0.jsonl'
    GC_URL = 'https://raw.githubusercontent.com/eps-research/rag-corpus-series/9734a42a025900bc26ec69373628114841e2b2e0/examples/gc/harris_gc_corpus_v1.4.0.jsonl'
    if not os.path.exists(GC_PATH):
        print(f"Downloading {GC_PATH}...")
        urllib.request.urlretrieve(GC_URL, GC_PATH)
        print(f"  ✓ {GC_PATH}")
    else:
        print(f"  Already present: {GC_PATH}")
    print("Ready.")
else:
    GC_PATH = 'harris_gc_corpus_v1.4.0.jsonl'
    print(f"Running locally — GC corpus: {GC_PATH}")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

clusters = []
with open(GC_PATH) as f:
    for line in f:
        clusters.append(json.loads(line))
print(f"Total clusters: {len(clusters)}")


In [ ]:
data=[]
for c in clusters:
    dh=c.get('distances',{}).get('r_sun_kpc')
    db=c.get('baumgardt2023',{}).get('r_sun_kpc') if c.get('baumgardt2023') else None
    if dh and db:
        data.append({'cluster':c['cluster_id'],'harris':dh,'baumgardt':db,'diff':db-dh})
diffs=[d['diff'] for d in data]
print(f"Clusters with both distances: {len(data)}")
print(f"Mean offset (Baumgardt - Harris): {np.mean(diffs):.2f} kpc")
print(f"Std: {np.std(diffs):.2f} kpc")
dh=[d['harris'] for d in data]; db=[d['baumgardt'] for d in data]
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(dh,db,s=15,alpha=0.6,color='#2166ac')
lim=[0,max(dh+db)*1.05]; axes[0].plot(lim,lim,'k--',lw=1,alpha=0.4)
axes[0].set_xlabel('Harris distance (kpc)',fontsize=11); axes[0].set_ylabel('Baumgardt distance (kpc)',fontsize=11)
axes[0].set_title('Harris vs Baumgardt Distances',fontsize=10)
axes[1].hist(diffs,bins=20,color='#d62728',alpha=0.8,edgecolor='white')
axes[1].axvline(0,color='black',ls='--',lw=1.5)
axes[1].set_xlabel('Baumgardt - Harris (kpc)',fontsize=11); axes[1].set_title('Distance Offset',fontsize=10)
plt.suptitle('Distance Comparison — EPS Research GC Corpus v1.4.0',fontsize=11)
plt.tight_layout(); plt.savefig('gc17_distances.png',dpi=150,bbox_inches='tight'); plt.show()